# Tree of Thoughts for Automated Code Debugging
**CS 4782 Final Project — Cornell University**

This notebook runs the full experiment pipeline:
1. Install Ollama + pull Qwen2.5-Coder-7B (free, open-source)
2. Load HumanEval-Bugs dataset
3. Run ToT-BFS, ToT-DFS, and baselines (IO, CoT, CoT-SC)
4. Display results table and comparison plots

> **Runtime**: Set to GPU (T4) for faster inference. Runtime → Change runtime type → T4 GPU.

## Step 1: Install Dependencies

In [ ]:
!pip install -q openai datasets matplotlib numpy tqdm

## Step 2: Install Ollama and Pull Qwen2.5-Coder-7B
This takes ~3-5 minutes to download the model (~4.7 GB). Run once per session.

In [ ]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time

# Start Ollama server in background
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)
print("Ollama server started.")

In [ ]:
# Pull the model — this is the slow step (~5 min on Colab)
!ollama pull qwen2.5-coder:7b

In [ ]:
# Verify model is available
!ollama list

## Step 3: Clone Repo and Set Up Paths

In [ ]:
!git clone https://github.com/jeffyz805/cs4782_final_project.git

import sys
sys.path.insert(0, '/content/cs4782_final_project/code')

import os
os.chdir('/content/cs4782_final_project')
print("Repo ready. Working directory:", os.getcwd())

## Step 4: Quick Sanity Check (Demo Mode)

In [ ]:
# Test that the LLM client can reach Ollama
import sys
sys.path.insert(0, 'code')

from llm_client import LLMClient
llm = LLMClient()  # defaults to ollama + qwen2.5-coder:7b
resp = llm.call("Say hello in one sentence.", max_tokens=50)
print("Model response:", resp['content'])
print("Tokens used:", resp['tokens'])

## Step 5: Load Dataset
Downloads HumanEval from HuggingFace and introduces synthetic bugs.

In [ ]:
from data_loader import load_humaneval_bugs

NUM_PROBLEMS = 50

problems = load_humaneval_bugs(n=NUM_PROBLEMS, use_synthetic=False, seed=42)
print(f"Loaded {len(problems)} problems")

# Bug type distribution
from collections import Counter
bug_types = Counter(p.bug_type for p in problems)
for bt, count in bug_types.most_common():
    print(f"  {bt:25s}: {count}")

In [ ]:
# Preview one problem
p = problems[0]
print(f"Task ID  : {p.task_id}")
print(f"Bug type : {p.bug_type}")
print(f"\n--- Buggy code ---\n{p.buggy_code}")
print(f"\n--- Test code ---\n{p.test_code[:300]}")

## Step 6: Run All Experiments
Runs ToT-BFS, ToT-DFS, IO, CoT, and CoT-SC on all problems.

**Estimated time on Colab T4**: ~60-90 minutes total.

In [ ]:
from llm_client import LLMClient
from executor import CodeExecutor
from tot_debugger import ToTDebugger
from baselines import IOBaseline, CoTBaseline, CoTSCBaseline
from evaluate import save_results, compute_metrics
from tqdm.notebook import tqdm
import os

llm = LLMClient()
executor = CodeExecutor()
os.makedirs('results', exist_ok=True)

K = 3  # branching factor

def run_method(solver, problems, label):
    results = []
    for p in tqdm(problems, desc=label):
        try:
            r = solver.solve(p)
        except Exception as e:
            print(f"  ERROR {p.task_id}: {e}")
            from tot_debugger import DebugResult
            r = DebugResult(p.task_id, label, False, None, 0, 0, 0, 0.0, False, p.bug_type)
        results.append(r)
    m = compute_metrics(results)
    print(f"{label}: fix_rate={m['fix_rate']:.1%}  avg_tokens={m['avg_tokens']:.0f}")
    return results

all_results = {}

In [ ]:
# ToT - BFS
solver = ToTDebugger(llm, executor, k=K, search='bfs', evaluator='hybrid')
all_results['tot-bfs-k3'] = run_method(solver, problems, 'tot-bfs-k3')
save_results(all_results['tot-bfs-k3'], 'results/tot-bfs-k3.json')

In [ ]:
# ToT - DFS
solver = ToTDebugger(llm, executor, k=K, search='dfs', evaluator='hybrid')
all_results['tot-dfs-k3'] = run_method(solver, problems, 'tot-dfs-k3')
save_results(all_results['tot-dfs-k3'], 'results/tot-dfs-k3.json')

In [ ]:
# Baseline: IO
all_results['io'] = run_method(IOBaseline(llm, executor), problems, 'io')
save_results(all_results['io'], 'results/io.json')

In [ ]:
# Baseline: CoT
all_results['cot'] = run_method(CoTBaseline(llm, executor), problems, 'cot')
save_results(all_results['cot'], 'results/cot.json')

In [ ]:
# Baseline: CoT-SC (5 samples)
all_results['cot-sc-5'] = run_method(CoTSCBaseline(llm, executor, n_samples=5), problems, 'cot-sc-5')
save_results(all_results['cot-sc-5'], 'results/cot-sc-5.json')

## Step 7: Results Table

In [ ]:
import json
from evaluate import compare_methods, print_comparison_table, compute_by_bug_type

comparison = compare_methods(all_results)
print_comparison_table(comparison)

# Save summary
with open('results/summary.json', 'w') as f:
    json.dump(comparison, f, indent=2)
print("\nSummary saved to results/summary.json")

In [ ]:
# Bug-type breakdown for ToT-BFS
print("\n── Bug-type breakdown (tot-bfs-k3) ──")
by_type = compute_by_bug_type(all_results['tot-bfs-k3'])
for bt, m in sorted(by_type.items()):
    bar = '█' * int(m['fix_rate'] * 20)
    print(f"  {bt:25s} {bar:<20s} {m['fix_rate']:.1%}  (n={m['n']})")

## Step 8: Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

methods = list(comparison.keys())
fix_rates   = [comparison[m]['fix_rate'] * 100 for m in methods]
first_rates = [comparison[m].get('first_attempt_rate', 0) * 100 for m in methods]

x = np.arange(len(methods))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, fix_rates,   w, label='Fix Rate (%)',          color='#2196F3', alpha=0.9)
b2 = ax.bar(x + w/2, first_rates, w, label='1st-Attempt Rate (%)',  color='#FF9800', alpha=0.9)

ax.set_ylabel('Success Rate (%)', fontsize=13)
ax.set_title('Fix Rate by Method — HumanEval-Bugs (Qwen2.5-Coder-7B)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=15, ha='right', fontsize=11)
ax.set_ylim(0, 110)
ax.legend(fontsize=11)
ax.bar_label(b1, fmt='%.1f%%', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.1f%%', padding=3, fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
plt.savefig('results/fix_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/fix_rates.png")

In [ ]:
# Token cost plot
tokens = [comparison[m]['avg_tokens'] for m in methods]
colors = ['#9E9E9E', '#9E9E9E', '#9E9E9E', '#4CAF50', '#4CAF50']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(methods, tokens, color=colors[:len(methods)], alpha=0.85)
ax.set_ylabel('Avg Tokens per Problem', fontsize=13)
ax.set_title('Token Cost by Method', fontsize=14)
ax.set_xticklabels(methods, rotation=15, ha='right', fontsize=11)
ax.bar_label(bars, fmt='%.0f', padding=3, fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

baseline_patch = mpatches.Patch(color='#9E9E9E', label='Baseline')
tot_patch = mpatches.Patch(color='#4CAF50', label='ToT')
ax.legend(handles=[baseline_patch, tot_patch], fontsize=11)
fig.tight_layout()
plt.savefig('results/token_cost.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/token_cost.png")

In [ ]:
# Bug-type breakdown (horizontal bar)
by_type = compute_by_bug_type(all_results['tot-bfs-k3'])
bug_labels = list(by_type.keys())
bug_rates  = [by_type[bt]['fix_rate'] * 100 for bt in bug_labels]
bug_ns     = [by_type[bt]['n'] for bt in bug_labels]

palette = ['#4CAF50', '#2196F3', '#FF9800', '#F44336', '#9C27B0']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(bug_labels, bug_rates, color=palette[:len(bug_labels)], alpha=0.85)
ax.set_xlabel('Fix Rate (%)', fontsize=13)
ax.set_title('ToT-BFS Fix Rate by Bug Type', fontsize=14)
ax.set_xlim(0, 110)
for i, (v, n) in enumerate(zip(bug_rates, bug_ns)):
    ax.text(v + 1.5, i, f'{v:.1f}%  (n={n})', va='center', fontsize=10)
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
plt.savefig('results/bug_type_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/bug_type_breakdown.png")

In [ ]:
# Backtrack rate (DFS only)
dfs_results = all_results['tot-dfs-k3']
backtrack_counts = [r.backtracks for r in dfs_results]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Histogram of backtracks
axes[0].hist(backtrack_counts, bins=range(0, max(backtrack_counts)+2),
             color='#9C27B0', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Backtracks per Problem', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('DFS Backtrack Distribution', fontsize=13)
axes[0].grid(axis='y', alpha=0.3)

# Fix rate: problems that needed backtracking vs didn't
needed_bt  = [r for r in dfs_results if r.backtracks > 0]
no_bt      = [r for r in dfs_results if r.backtracks == 0]
labels = ['No backtrack', 'Backtracked']
rates  = [
    sum(r.success for r in no_bt)  / max(len(no_bt), 1) * 100,
    sum(r.success for r in needed_bt) / max(len(needed_bt), 1) * 100,
]
axes[1].bar(labels, rates, color=['#2196F3', '#FF5722'], alpha=0.85)
axes[1].set_ylabel('Fix Rate (%)', fontsize=12)
axes[1].set_title('DFS Fix Rate: Backtrack vs No Backtrack', fontsize=13)
axes[1].set_ylim(0, 110)
for i, v in enumerate(rates):
    axes[1].text(i, v + 2, f'{v:.1f}%', ha='center', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.savefig('results/backtrack_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/backtrack_analysis.png")

## Step 9: Save All Results and Download
Download the `results/` folder to use in your report and poster.

In [ ]:
import shutil

# Zip results folder for easy download
shutil.make_archive('/content/tot_results', 'zip', '/content/cs4782_final_project/results')
print("Results zipped → /content/tot_results.zip")

# Download (Colab only)
try:
    from google.colab import files
    files.download('/content/tot_results.zip')
except ImportError:
    print("Not running on Colab — find the zip at /content/tot_results.zip")

In [ ]:
# Final summary printout
print("=" * 55)
print(f"{'Method':<18} {'Fix Rate':>10} {'Avg Tokens':>12}")
print("=" * 55)
for method, m in comparison.items():
    print(f"{method:<18} {m['fix_rate']:>9.1%} {m['avg_tokens']:>12.0f}")
print("=" * 55)